# VCClone Colab demo — OpenCV fallback

This notebook runs a lightweight Flask inference server in Colab using OpenCV's Haar cascade to detect faces and a simple affine warp of a reference image. It exposes `/infer` via ngrok so you can point the browser client to it. This avoids MediaPipe installation problems and runs reliably in Colab.

Usage: Runtime -> Change runtime type -> GPU (not required for this demo). Run cells in order.

In [ ]:
# Install dependencies
!pip install -q flask flask_cors pyngrok opencv-python-headless pillow numpy

In [ ]:
# (Optional) clone the repo into the Colab workspace so app can reuse assets
!git clone https://github.com/MATASAKA12/vcclone.git || true
%cd vcclone || true

In [ ]:
%%bash
cat > app.py << 'PY'
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import base64, io
from PIL import Image, ImageDraw
import numpy as np
import cv2

app = Flask(__name__)
CORS(app)

# Create a simple reference face image
REF_W, REF_H = 400, 400
def make_reference_face():
    img = Image.new('RGBA', (REF_W, REF_H), (242,210,201,255))
    draw = ImageDraw.Draw(img)
    draw.ellipse((110,140,150,180), fill=(59,59,59))
    draw.ellipse((250,140,290,180), fill=(59,59,59))
    draw.ellipse((128,152,136,160), fill=(255,255,255))
    draw.ellipse((268,152,276,160), fill=(255,255,255))
    draw.arc((140,230,260,290), start=0, end=180, fill=(59,59,59), width=8)
    return img.convert('RGB')

reference_img = np.array(make_reference_face())

# Haar cascade face detector (bundled with OpenCV)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def decode_base64_image(data_url):
    header, encoded = data_url.split(',', 1) if ',' in data_url else (None, data_url)
    data = base64.b64decode(encoded)
    img = Image.open(io.BytesIO(data)).convert('RGB')
    return np.array(img)

def encode_image_to_base64(img_np):
    pil = Image.fromarray(img_np)
    buf = io.BytesIO()
    pil.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

@app.route('/infer', methods=['POST'])
def infer():
    data = request.get_json(force=True)
    if not data or 'image' not in data:
        return jsonify({'error': 'no image'}), 400
    try:
        frame = decode_base64_image(data['image'])
        h, w = frame.shape[:2]
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60,60))
        if len(faces) == 0:
            out = cv2.resize(reference_img, (w, h))
            return jsonify({'image': encode_image_to_base64(out)})
        # pick the largest face
        faces = sorted(faces, key=lambda r: r[2]*r[3], reverse=True)
        x,y,fw,fh = faces[0]
        # approximate left eye, right eye, nose positions from bbox
        left_eye = (int(x + fw*0.28), int(y + fh*0.35))
        right_eye = (int(x + fw*0.72), int(y + fh*0.35))
        nose = (int(x + fw*0.5), int(y + fh*0.55))
        dst = np.array([left_eye, right_eye, nose], dtype=np.float32)
        # source points in the reference image (tuned heuristics)
        src = np.array([[130,160],[270,160],[200,190]], dtype=np.float32)
        M = cv2.getAffineTransform(src, dst)
        warped = cv2.warpAffine(reference_img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        return jsonify({'image': encode_image_to_base64(warped)})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

def run_server():
    public_url = ngrok.connect(5000, bind_tls=True).public_url
    print(' * ngrok url:', public_url)
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False, threaded=True)

if __name__ == '__main__':
    run_server()
PY

In [ ]:
# Run the Flask app in the background so the notebook remains interactive
import subprocess, time
p = subprocess.Popen(['python3','app.py'])
time.sleep(2)
print('Started app.py in background. Check output above for ngrok URL.')


In [ ]:
# Quick test: find ngrok tunnel and POST a sample image
from pyngrok import ngrok
import requests, base64, io
from PIL import Image
tunnels = ngrok.get_tunnels()
if len(tunnels) > 0:
    public_url = tunnels[0].public_url
    infer_url = public_url + '/infer'
    print('Using', infer_url)
    img = Image.new('RGB', (480, 360), (120,160,200))
    buf = io.BytesIO()
    img.save(buf, format='JPEG')
    data_url = 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')
    resp = requests.post(infer_url, json={'image': data_url})
    print('Status:', resp.status_code)
    if resp.ok and 'image' in resp.json():
        out = resp.json().get('image')
        img_data = base64.b64decode(out.split(',',1)[1])
        open('sample_out.png','wb').write(img_data)
        print('Saved sample_out.png')
else:
    print('No ngrok tunnel found. Check app.py output for ngrok URL.')


### Notes
- This demo uses OpenCV Haar cascades for face detection which is less precise than MediaPipe landmarks but is reliable and avoids heavy dependencies.
- After the app starts, copy the printed ngrok URL and paste it into your browser client `public/client.js` as the `INFER_URL` value, or paste it here and I can commit it for you.
